<a href="https://colab.research.google.com/github/carolmesq/treinamento_python/blob/main/Aula4_Bloco3_Python_aplicado_geo_github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise da rede de transporte da RMSP

Manipulação de dados para gerenciar e consultar dadoms urbanos e geoespaciais utilizando Python.

In [ ]:
# Instalar as bibliotecas
!pip install pandas
!pip install geopandas
!pip install numpy

In [ ]:
# Importar as bibliotecas
import pandas as pd
import geopandas as gpd
import numpy as np

In [ ]:
from google.colab import files

# 1. Fazer upload do arquivo .gpkg
uploaded = files.upload()
filename = list(uploaded.keys())[0]  # Nome do arquivo enviado

Saving cenario302_link.gpkg to cenario302_link.gpkg


In [ ]:
# 2. Carregar o GeoPackage
#    O comando read_file() lê a primeira camada por padrão.
gdf = gpd.read_file(filename)

# 3. Visualizar as primeiras linhas
print(gdf.head())

# 4. Informações gerais (tipos, valores nulos, geometria)
print(gdf.info())

      id     NO  FROMNODENO  TONODENO TYPENO  \
0  67276  33648      113849    100197    105   
1  67277  33649      100198    100199      0   
2  67278  33649      100199    100198      0   
3  67279  33650      100198    100297    105   
4  67280  33650      100297    100198      0   

                                             TSYSSET LENGTH  NUMLANES  CAPPRT  \
0  BRT-ABC,BRT-AlphCaj,BRT-Aricand,BRT-Cajamar,BR...     84         4    4800   
1                                                  p     56         1       0   
2                                                  p     56         1       0   
3  BRT-ABC,BRT-AlphCaj,BRT-Aricand,BRT-Cajamar,BR...    411         4    4800   
4                                                  p    411         4       0   

  V0PRT  ...  VOL_MOD~35  VOL_OBS~36  VOL_OBS~37  VOL_ONIBUS  VOL_PRE~38  \
0    45  ...           0          16           0          16           0   
1     0  ...           0           0           0           0           0

In [ ]:
# 5. Converter nomes das colunas para minúsculas
gdf.columns = gdf.columns.str.lower()

# 6. (Opcional) Verificar novamente as informações após a alteração
print(gdf.info())

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 161498 entries, 0 to 161497
Data columns (total 76 columns):
 #   Column      Non-Null Count   Dtype   
---  ------      --------------   -----   
 0   id          161498 non-null  int32   
 1   no          161498 non-null  int32   
 2   fromnodeno  161498 non-null  int32   
 3   tonodeno    161498 non-null  int32   
 4   typeno      161498 non-null  object  
 5   tsysset     160281 non-null  object  
 6   length      161498 non-null  object  
 7   numlanes    161498 non-null  int32   
 8   capprt      161498 non-null  int32   
 9   v0prt       161498 non-null  object  
 10  volvehpr~1  161498 non-null  int32   
 11  volpersp~2  161498 non-null  int32   
 12  volpersw~3  161498 non-null  int32   
 13  acesso      161496 non-null  float64 
 14  cap_ajus~4  161498 non-null  int32   
 15  cap_ajus~5  161498 non-null  int32   
 16  configur~6  7074 non-null    object  
 17  corredores  161498 non-null  object  
 18  corredor~7  1614

Converter a coluna typeno para inteiro

In [ ]:
# 7. Converte para numérico, transformando valores não numéricos em NaN
gdf['typeno'] = pd.to_numeric(gdf['typeno'], errors='coerce')

# 8. Preenche os NaN com 0 e converte para inteiro
gdf['typeno'] = gdf['typeno'].fillna(0).astype(int)

7. **pd.to_numeric()** função do pandas de converter os valores de uma série (ou lista) para números (int ou float).


* o parâmetro **errors='coerce'** define o comportamento para valores que não podem ser convertidos: em vez de gerar um erro, esses valores são transformados em NaN (Not a Number). Isso é útil quando a coluna contém dados inconsistentes, como strings, textos ou valores ausentes que deveriam ser números.

8. **gdf['typeno'].fillna(0)** – substitui todos os valores NaN (que foram gerados na conversão anterior quando algo não era numérico) pelo número 0.

In [ ]:
# 9. Converter length para numérico
gdf['length'] = gdf['length'].astype(int)

print(gdf['length'].dtype)

int64


## Manipulação da tabela de atributos

### 1. Criar a coluna **vdf** (*volume delay function*) baseada na coluna **typeno**

Tipo link - Classe - VDF
* 100 - Rodovia Estrutural 1.1 - 1
* 101 - Rodovia Estrutural 1.2 - 1
* 102 - Rodovia Estrutural 2.1 - 1
* 104 - Arterial I - 2
* 105 - Arterial II - 3
* 106 - Coletora - 4
* 107 - Local - 5

In [ ]:
# 1. Mapeamento de typeno para vdf
mapeamento_vdf = {
    100: 1,
    101: 1,
    102: 1,
    104: 2,
    105: 3,
    106: 4,
    107: 5,
    0: 0
}

# 2. Criar a nova coluna 'vdf' aplicando o mapeamento
# é um método usado em Series do pandas para mapear (substituir) cada valor da série com base em uma regra ou correspondência
gdf['vdf'] = gdf['typeno'].map(mapeamento_vdf)

# Caso haja algum valor não mapeado, preencher com 0 (ou outro valor padrão)
gdf['vdf'] = gdf['vdf'].fillna(0).astype(int)

# 3. Visualizar as primeiras linhas para conferir
print(gdf[['typeno', 'vdf']].head())

   typeno  vdf
0     105    3
1       0    0
2       0    0
3     105    3
4       0    0


### 2. Classificar **typeno** nova coluna "mode" com **p, r, m, t, v e cbekp**.

Condições:
* **typeno** igual a 0 e **tsysset** igual a 'p';
* **typeno** igual a 0 e **tsysset** igual a 'r';
* **typeno** igual a 202 e **tsysset** começa com a letra 'm';
* **typeno** igual a 202 e **tsysset** começa com 't';
* **typeno** igual a 202 e **tsysset** começa com 'v';
* **typeno** pertence a uma lista de valores específicos (de 100 a 107).


Classificação dos modos:

* p - a pé
* r - acesso metrô e trem
* m - metrô
* t - cptm
* v - vlt
* c - carro
* b - sptrans
* e - emtu
* k - caminhão

In [ ]:
# 1. Condições para preenchimento da coluna 'mode'
condicoes = [
    (gdf['typeno'] == 0) & (gdf['tsysset'] == 'p'),
    (gdf['typeno'] == 0) & (gdf['tsysset'] == 'r'),
    (gdf['typeno'] == 202) & (gdf['tsysset'].str.startswith('m', na=False)),
    (gdf['typeno'] == 202) & (gdf['tsysset'].str.startswith('t', na=False)),
    (gdf['typeno'] == 202) & (gdf['tsysset'].str.startswith('v', na=False)),
    gdf['typeno'].isin([100, 101, 102, 103, 104, 105, 106, 107])
]

valores = ['p', 'r', 'm', 't', 'v', 'cbekp']


# 2. Criar a coluna 'mode' aplicando as condições
gdf['mode'] = np.select(condicoes, valores, default='')

# 3. Verificar as primeiras linhas
print(gdf[['typeno', 'tsysset', 'mode']].head())

   typeno                                            tsysset   mode
0     105  BRT-ABC,BRT-AlphCaj,BRT-Aricand,BRT-Cajamar,BR...  cbekp
1       0                                                  p      p
2       0                                                  p      p
3     105  BRT-ABC,BRT-AlphCaj,BRT-Aricand,BRT-Cajamar,BR...  cbekp
4       0                                                  p      p


1. **.str.startswith('m', na=False)** verifica se o valor da string começa a letra determinada (ex. 'm'). O parâmetro **na=False** faz com que valores NaN sejam tratados como False (não geram erro).

O método **.isin()** retorna True para linhas onde o valor está presente na lista.

2. **np.select** (do NumPy) avalia cada linha e verifica qual condição da lista é a primeira a ser True.

* Se uma condição for verdadeira, atribui o valor correspondente da lista valores;
* Se nenhuma condição for verdadeira, atribui o valor definido em default (nesse caso, uma string vazia ' ');
* O resultado é uma nova coluna 'mode' no GeoDataFrame.

### 3. Extrair os nomes dos municípios da coluna **tsysset**

Extrair o nome do município contido na coluna tsysset, ignorando prefixos fixos e caracteres indesejados.Prefixo do texto 'MuniAux_nomedomunicipio.'

A biblioteca **re** no Python é o módulo para trabalhar com Expressões Regulares (Regex), permitindo buscar, validar, dividir ou substituir padrões de texto complexos em strings.

Documentação: https://docs.python.org/pt-br/3/library/re.html

In [ ]:
import re

# Dicionário de correção para nomes brutos que não são corretamente tratados pela lógica geral
correcoes_brutas = {
    'Diadema': 'Diadema',
    'EmbudasArtes': 'Embu das Artes',
    'MogidasCruzes': 'Mogi das Cruzes',
    'SãoBernardodoCampo': 'São Bernardo do Campo',
    # Adicione outras exceções se necessário
}

def extrair_municipio(valor):
    """Extrai e formata o nome do município, com correções manuais."""
    if pd.isna(valor):
        return ''
    if 'MuniAux_' not in valor:
        return ''    # Se o valor for NaN ou não contiver a substring 'MuniAux_', a função retorna uma string vazia.
    match = re.search(r'MuniAux_([^,]+)', valor) #A expressão busca o padrão MuniAux_ seguido de qualquer caractere que não seja vírgula ([^,]+).
    if not match:
        return ''
    nome_bruto = match.group(1)

    # Se o nome bruto estiver no dicionário de correções, retorna diretamente o nome corrigido
    if nome_bruto in correcoes_brutas:
        return correcoes_brutas[nome_bruto]

    # Caso contrário, aplica a lógica geral de formatação
    # Inserir espaço antes de letras maiúsculas
    nome_espacado = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', nome_bruto)

    # Separar preposições grudadas
    preposicoes = ['de', 'da', 'do', 'das', 'dos']
    for prep in preposicoes:
        padrao = re.compile(r'([a-záàâãéêíóôúç])({})(' r'[A-ZÁÀÂÃÉÊÍÓÔÚÇ]|\s)'.format(prep), re.IGNORECASE)
        nome_espacado = padrao.sub(r'\1 \2 \3', nome_espacado)

    # Divide a string em palavras. Capitalizar: primeira letra maiúscula, preposições em minúscula
    palavras = nome_espacado.split()
    palavras_corrigidas = []
    for palavra in palavras:
        if palavra.lower() in preposicoes:
            palavras_corrigidas.append(palavra.lower())
        else:
            palavras_corrigidas.append(palavra[0].upper() + palavra[1:])
    return ' '.join(palavras_corrigidas)

# Aplica a função na coluna 'tsysset' e cria a coluna 'auxiliar'
gdf['auxiliar'] = gdf['tsysset'].apply(extrair_municipio)

# Verifica os resultados para os casos problemáticos
print(gdf[gdf['tsysset'].str.contains('MuniAux_(Diadema|EmbudasArtes|MogidasCruzes)', na=False)][['tsysset', 'auxiliar']])

                                                  tsysset auxiliar
658     c,c_Compras,c_Educ,c_Outros,c_Trab1,c_Trab2,c_...  Diadema
659     c,c_Compras,c_Educ,c_Outros,c_Trab1,c_Trab2,c_...  Diadema
660     c,c_Compras,c_Educ,c_Outros,c_Trab1,c_Trab2,c_...  Diadema
661     c,c_Compras,c_Educ,c_Outros,c_Trab1,c_Trab2,c_...  Diadema
1930    BRT-ABC,BRT-AlphCaj,BRT-Aricand,BRT-Cajamar,BR...  Diadema
...                                                   ...      ...
159878  BRT-ABC,BRT-AlphCaj,BRT-Aricand,BRT-Cajamar,BR...  Diadema
159879  BRT-ABC,BRT-AlphCaj,BRT-Aricand,BRT-Cajamar,BR...  Diadema
159881  BRT-ABC,BRT-AlphCaj,BRT-Aricand,BRT-Cajamar,BR...  Diadema
159882  BRT-ABC,BRT-AlphCaj,BRT-Aricand,BRT-Cajamar,BR...  Diadema
159883  BRT-ABC,BRT-AlphCaj,BRT-Aricand,BRT-Cajamar,BR...  Diadema

[1458 rows x 2 columns]


/tmp/ipykernel_15059/4252711038.py:51: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  print(gdf[gdf['tsysset'].str.contains('MuniAux_(Diadema|EmbudasArtes|MogidasCruzes)', na=False)][['tsysset', 'auxiliar']])


Fazer uma contagem do total de trechos utilizando python

In [ ]:
contagem = gdf['auxiliar'].value_counts().reset_index()
contagem.columns = ['municipio', 'total_trechos']
print(contagem.head())

               municipio  total_trechos
0                                141955
1              Guarulhos           3544
2            Santo André           1874
3  São Bernardo do Campo           1723
4                 Osasco           1415


## SQL

### Fazer consultas utilizando bibliotecas que fazem leitura de SQL

In [ ]:
!pip install duckdb
import duckdb

A biblioteca **DuckDB** para Python é um sistema de gerenciamento de banco de dados analítico, de código aberto e orientado a colunas. Ela se integra ao  Python, permitindo que os usuários executem consultas SQL diretamente em estruturas de dados locais, como DataFrames do Pandas, sem a necessidade de um servidor de banco de dados separado.

Documentação: https://duckdb.org/docs/current/

In [ ]:
print(gdf.info())

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 161498 entries, 0 to 161497
Data columns (total 79 columns):
 #   Column      Non-Null Count   Dtype   
---  ------      --------------   -----   
 0   id          161498 non-null  int32   
 1   no          161498 non-null  int32   
 2   fromnodeno  161498 non-null  int32   
 3   tonodeno    161498 non-null  int32   
 4   typeno      161498 non-null  int64   
 5   tsysset     160281 non-null  object  
 6   length      161498 non-null  int64   
 7   numlanes    161498 non-null  int32   
 8   capprt      161498 non-null  int32   
 9   v0prt       161498 non-null  object  
 10  volvehpr~1  161498 non-null  int32   
 11  volpersp~2  161498 non-null  int32   
 12  volpersw~3  161498 non-null  int32   
 13  acesso      161496 non-null  float64 
 14  cap_ajus~4  161498 non-null  int32   
 15  cap_ajus~5  161498 non-null  int32   
 16  configur~6  7074 non-null    object  
 17  corredores  161498 non-null  object  
 18  corredor~7  1614

In [ ]:
print(gdf.columns.tolist())

['id', 'no', 'fromnodeno', 'tonodeno', 'typeno', 'tsysset', 'length', 'numlanes', 'capprt', 'v0prt', 'volvehpr~1', 'volpersp~2', 'volpersw~3', 'acesso', 'cap_ajus~4', 'cap_ajus~5', 'configur~6', 'corredores', 'corredor~7', 'criado', 'desativar', 'descricao', 'emtu', 'estrategia', 'faixa_ex~8', 'fator_ve~9', 'fx_excl~10', 'hub_perif', 'hub_sp', 'lengthc~11', 'links_s~12', 'miniane~13', 'mun', 'nomelinha', 'nome_in~14', 'nrotas', 'numlane~15', 'oferta_put', 'perif', 'pesos', 'peso_ce~16', 'peso_fx~17', 'prop_corr', 'prop_tr~18', 'ref_cal~19', 'sentido~20', 'sistema', 'sp', 'sptrans', 'tarifa_~21', 'temporotas', 'tempotr~22', 'tempotr~23', 'tempotr~24', 'tempotr~25', 'tipocor~26', 'trilhos', 'tsysaux~27', 'vcur_pr~28', 'veloc_c~29', 'veloc_r~30', 'velsptr~31', 'vel_cap', 'vel_cor~32', 'vol_fix~33', 'vol_fix~34', 'vol_mod~35', 'vol_obs~36', 'vol_obs~37', 'vol_onibus', 'vol_pre~38', 'vpapm', 'vpapm_emme', 'vpapm_s~39', 'zmrc', 'geometry', 'vdf', 'mode', 'auxiliar']


In [ ]:
# 1. Remove temporariamente a coluna geometry para a consulta
gdf_sem_geom = gdf.drop(columns=['geometry'])

# 2. Consulta as estatísticas (sem geometry)
estatisticas = duckdb.sql("""
    SELECT
        auxiliar,
        COUNT(*) AS total_trechos,
        SUM(length) AS comprimento_total
    FROM gdf_sem_geom
    GROUP BY auxiliar
    ORDER BY total_trechos DESC
""").df()

# 3. Mescla as estatísticas de volta ao GeoDataFrame original
gdf_com_estatisticas = gdf.merge(estatisticas, on='auxiliar', how='left')

# Agora imprima apenas as estatísticas, não o gdf_sem_geom
print(estatisticas.head(32))

                 auxiliar  total_trechos  comprimento_total
0                                 141955         23499432.0
1               Guarulhos           3544          1122321.0
2             Santo André           1874           445853.0
3   São Bernardo do Campo           1723           659762.0
4                  Osasco           1415           424157.0
5                 Barueri           1394           401997.0
6                    Mauá            815           284517.0
7                Caieiras            804           218191.0
8        Francisco Morato            740           195194.0
9         Mogi das Cruzes            701           581825.0
10                Diadema            651           180670.0
11   Itapecerica da Serra            506           215735.0
12                  Cotia            475           467889.0
13        Itaquaquecetuba            443           207545.0
14        Franco da Rocha            435           208273.0
15                  Arujá            427

### Exportar no formato geopackage

In [ ]:
gdf.to_file('rede01_vdf.gpkg', driver='GPKG')